# Phase E — Eval results

Compare three policies on `Billiards4BallEnv` (1000 episodes each):

1. `random`     — uniform sampling on the action space.
2. `geometric`  — deterministic aim-at-nearest-red baseline.
3. `ppo`        — PPO trained against the learned reward model.

Reads the parquet emitted by `scripts/eval_policies.py`. The last
section re-runs the best three PPO scoring episodes and renders them
inline as HTML replays.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

from billiards.env import Billiards4BallEnv
from billiards.render.replay import render_html

PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_PATH = PROJECT_ROOT / 'data' / 'eval_results.parquet'
PPO_PATH = PROJECT_ROOT / 'models' / 'ppo_policy.zip'
print('results parquet:', RESULTS_PATH, 'exists =', RESULTS_PATH.exists())
print('ppo policy:    ', PPO_PATH, 'exists =', PPO_PATH.exists())

In [ ]:
df = pd.read_parquet(RESULTS_PATH)
print('rows:', len(df), 'policies:', sorted(df['policy'].unique()))
df.head()

## Comparison table

In [ ]:
summary = (
    df.groupby('policy')
      .agg(
          n=('score', 'size'),
          score_rate=('score', 'mean'),
          foul_rate=('fouled', 'mean'),
          mean_cushions=('cushion_hits', 'mean'),
          mean_duration=('duration', 'mean'),
      )
      .reset_index()
)
summary['score_rate'] = (100.0 * summary['score_rate']).round(2)
summary['foul_rate'] = (100.0 * summary['foul_rate']).round(2)
summary['mean_cushions'] = summary['mean_cushions'].round(2)
summary['mean_duration'] = summary['mean_duration'].round(2)
summary

## Score rate by policy

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
order = ['random', 'geometric', 'ppo']
subset = summary.set_index('policy').reindex([p for p in order if p in summary['policy'].values])
ax.bar(subset.index, subset['score_rate'])
ax.set_ylabel('Score rate (%)')
ax.set_title('Carom score rate over 1000 evaluation episodes')
for x, v in zip(subset.index, subset['score_rate']):
    ax.text(x, v + 0.5, f'{v:.1f}%', ha='center')
fig.tight_layout()
plt.show()

## Best 3 PPO scoring episodes (cleanest = lowest cushions)

Filter PPO rows where `score == 1`, sort by ascending `cushion_hits`,
take the top 3, re-run the env on those seeds (deterministic predict),
and render the trajectories inline.

In [ ]:
ppo_rows = df[df['policy'] == 'ppo']
scoring = ppo_rows[ppo_rows['score'] == 1].sort_values(['cushion_hits', 'duration']).head(3)
print(f'PPO scoring episodes available: {(ppo_rows["score"] == 1).sum()}')
scoring

In [ ]:
if PPO_PATH.exists() and len(scoring) > 0:
    from stable_baselines3 import PPO
    ppo_model = PPO.load(str(PPO_PATH), device='cpu')
    for _, row in scoring.iterrows():
        env = Billiards4BallEnv()
        obs, info0 = env.reset(seed=int(row['seed']))
        action, _ = ppo_model.predict(obs, deterministic=True)
        _, _, _, _, info = env.step(action)
        html = render_html(
            info['trajectory'],
            spec=info0.get('spec'),
        )
        print(
            f"seed={int(row['seed'])} score={info['score']} cushions={info['cushion_hits']} "
            f"duration={info['duration']:.2f}s"
        )
        display(HTML(html if isinstance(html, str) else html.data))
else:
    print('No PPO policy or no scoring episodes — skipping inline replays.')

## Verdict

In [ ]:
by_policy = summary.set_index('policy')['score_rate'].to_dict()
rand_rate = by_policy.get('random', float('nan'))
geo_rate = by_policy.get('geometric', float('nan'))
ppo_rate = by_policy.get('ppo', float('nan'))
lift_vs_rand = ppo_rate - rand_rate if ppo_rate == ppo_rate else float('nan')
lift_vs_geo = ppo_rate - geo_rate if ppo_rate == ppo_rate else float('nan')
print(
    f'VERDICT — random={rand_rate:.1f}% geometric={geo_rate:.1f}% ppo={ppo_rate:.1f}% '
    f'(PPO lift: +{lift_vs_rand:.1f}pp vs random, {lift_vs_geo:+.1f}pp vs geometric)'
)